## 2.6 文本预处理 - 有效长度、padding 干扰与为什么需要 mask 或 pack_padded_sequence

#### 1、为什么这一小节必须紧接着学习

##### 1.1 上一节我们解决了“形状统一”的问题，但还没有解决“无效位置干扰”的问题
上一节我们已经知道：

- 不同句子长度不同，不能直接组成 batch
- 为了 batch 训练，通常要做 padding 和 truncation
- 做完之后，句子可以整理成统一长度的张量
- 然后再经过 Embedding，送进 RNN

例如一个 batch 中可能出现这样的序列：

```python
[2, 3, 4, 0, 0]
[5, 6, 7, 8, 9]
[10, 11, 0, 0, 0]
```

从形状上看，它们已经很整齐了，都长度为 `5`。

但是新的问题也出现了：

这些 `0`，也就是 `PAD`，根本不是真实单词。  
可如果我们直接把整个序列送进 RNN，RNN 仍然会把这些 `PAD` 位置也当成时间步去计算。

所以，上一节解决的是：

“怎么把 batch 拼起来”

这一节要解决的是：

“拼起来之后，模型怎么分清哪些位置是真的，哪些位置只是 padding”

##### 1.2 统一长度不等于真实长度相同
例如下面两个序列：

```python
[2, 3, 4, 0, 0]
[5, 6, 7, 8, 9]
```

它们的张量长度都等于 `5`，  
但真实有效长度并不一样：

- 第一条的真实长度是 `3`
- 第二条的真实长度是 `5`

所以我们必须区分两个概念：

- 张量长度，也就是 padding 后的统一长度
- 有效长度，也就是真实 token 的数量

这一小节的核心，就是建立这个区别。

##### 1.3 这一节是“变长序列真正进入 RNN”的关键
如果只学到 padding，我们只是学会了“外形统一”。

但真实的文本序列处理还需要进一步考虑：

- 如何避免 `PAD` 干扰隐藏状态
- 如何避免无效计算
- 如何让模型真正只关注有效 token

所以这一节是从“能跑起来”走向“更合理地跑”的关键一步。


#### 2、什么叫有效长度

##### 2.1 有效长度的定义
有效长度，指的是：

一个序列中真实 token 的数量，不包括后面补上的 `PAD`。

例如：

```python
[2, 3, 4, 0, 0]
```

这个序列虽然总长度是 `5`，  
但其中只有前 `3` 个位置是真实 token，后面两个 `0` 只是补齐位置。

所以它的：

- 张量长度是 `5`
- 有效长度是 `3`

##### 2.2 为什么要单独强调“有效长度”
因为模型表面看到的是统一长度序列，  
但从语义角度看，真正有意义的部分只有前面那些真实 token。

例如：

`I love AI`

$\rightarrow$

```python
[2, 3, 4, 0, 0]
```

这句话真正的内容只到第 `3` 个位置就结束了。  
后面的 `PAD` 不应该继续影响句子的表示。

所以我们必须把“有效长度”单独记录下来，告诉模型：

这个样本真正读到哪里就该停了。

##### 2.3 再看几个例子
序列 A：

```python
[2, 3, 4, 0, 0]
```

有效长度 $= 3$

序列 B：

```python
[5, 6, 7, 8, 9]
```

有效长度 $= 5$

序列 C：

```python
[10, 11, 0, 0, 0]
```

有效长度 $= 2$

你会发现：

虽然这 `3` 条序列 padding 后长度都一样，  
但它们真正的有效信息长度完全不同。


#### 3、为什么 padding 会带来干扰

##### 3.1 RNN 默认并不知道 PAD 是“假的”
RNN 的工作方式是按时间步依次处理输入：

$x_1 \rightarrow x_2 \rightarrow x_3 \rightarrow x_4 \rightarrow x_5$

如果输入序列是：

```python
[2, 3, 4, 0, 0]
```

那么在 RNN 看来，它会照样处理 `5` 个时间步：

- 第 `1` 步：真实 token
- 第 `2` 步：真实 token
- 第 `3` 步：真实 token
- 第 `4` 步：`PAD`
- 第 `5` 步：`PAD`

也就是说，如果我们不额外处理，RNN 并不会自动知道：

第 `4`、`5` 步其实不是真实内容。

##### 3.2 PAD 也会参与前向传播和反向传播计算
RNN 模型中，每一个时间步都会影响下一个时间步。  
只要一个时间步有输入，RNN 就会继续计算新的 $h_t$。

这意味着：

即使第 `4`、`5` 个位置只是 `PAD`，  
它们仍然会继续更新隐藏状态。

这就可能导致一个问题：

原本前 `3` 个真实 token 得到的隐藏状态，  
会被后面无意义的 `PAD` 再“冲刷”几次。

这样最终得到的句子表示，就可能被污染。

##### 3.3 PAD 会带来无效计算
除了干扰语义外，padding 还会带来额外的计算成本。

例如一个 batch 中很多句子都很短，但为了统一长度都补到了 `50`。  
这意味着：

很多时间步其实都只是 `PAD`，  
但 RNN 仍然在这些位置上做前向传播和反向传播。

这会导致：

- 浪费计算资源
- 降低训练效率
- 让模型把精力分散到无效位置上

##### 3.4 长短句混合时，影响会更明显
例如一个 batch 中：

- 一条句子真实长度是 `48`
- 另一条句子真实长度是 `7`

如果统一补到 `50`，  
那条短句会有大量 `PAD`。

这时如果不做特殊处理，  
短句的后面几十步都在做无意义的 RNN 计算。

所以 padding 干扰在变长差异较大的 batch 中会更明显。

#### 4、padding 干扰具体表现在哪些地方

##### 4.1 干扰最终隐藏状态
在很多任务中，我们会把 RNN 最后一个时间步的隐藏状态，作为整句话的表示。

例如：

$h_T$ 作为句向量

但问题是，如果序列是：

```python
[2, 3, 4, 0, 0]
```

那么最后一个时间步对应的其实是 `PAD`，而不是最后一个真实词。

这就会导致：

你取到的“最后状态”，不是真正句子结束时的状态，  
而是又经过几次 `PAD` 更新之后的状态。

这显然不够合理。

##### 4.2 干扰序列输出
如果任务需要使用每个时间步的输出，例如：

$o_1, o_2, o_3, o_4, o_5$

那么 padding 位置对应的输出也会一起存在。

这时如果后续不区分有效位置和无效位置，  
就可能把 `PAD` 位置的输出也拿去做后续计算或损失统计。

这同样会带来问题。

##### 4.3 干扰损失计算
例如在序列标注或语言模型任务中，  
每个时间步都可能对应一个预测目标。

如果不做 mask，模型甚至可能在 `PAD` 位置上也计算 loss。

但 `PAD` 根本不是真实样本内容，  
它不应该参与损失。

所以 padding 不仅影响前向表示，  
还可能影响 loss 的正确计算。


#### 5、什么是 mask

##### 5.1 mask 的最直观理解
mask 可以先简单理解成：

一个“位置说明表”，专门告诉模型哪些位置是有效的，哪些位置是无效的。

例如序列：

```python
[2, 3, 4, 0, 0]
```

如果 `0` 是 `PAD`，那么对应的 mask 可以写成：

```python
[1, 1, 1, 0, 0]
```

这里的含义是：

- `1` 表示这个位置是真实 token
- `0` 表示这个位置是 padding

##### 5.2 mask 的作用
mask 的核心作用就是：

把“有效位置”和“无效位置”显式标记出来

这样后续模型或损失函数就可以知道：

- 哪些位置应该参与计算
- 哪些位置应该忽略

##### 5.3 再看几个例子
序列 A：

```python
[2, 3, 4, 0, 0]
```

mask：

```python
[1, 1, 1, 0, 0]
```

序列 B：

```python
[5, 6, 7, 8, 9]
```

mask：

```python
[1, 1, 1, 1, 1]
```

序列 C：

```python
[10, 11, 0, 0, 0]
```

mask：

```python
[1, 1, 0, 0, 0]
```

你会发现，mask 本质上就是把有效长度写成逐位置形式。

#### 6、什么是 pack_padded_sequence

##### 6.1 它不是“标记无效位置”，而是“直接跳过无效位置”
如果说 mask 的思路是：

先保留 padding 后的完整张量，  
再通过 `0` 和 `1` 告诉模型哪些位置有效

那么 `pack_padded_sequence` 的思路更进一步：

既然 padding 位置本来就没意义，  
那能不能在 RNN 内部干脆别算它们？

答案是：可以。  
这正是 `pack_padded_sequence` 的核心思想。

##### 6.2 它的本质理解
`pack_padded_sequence` 可以理解为：

把已经 padding 的 batch 序列，重新打包成“只保留有效部分”的紧凑表示，  
让 RNN 在内部只处理真实 token，而尽量跳过 `PAD` 对应的时间步。

也就是说：

- mask 更像“算完后忽略无效位置”
- pack 更像“从一开始就尽量不去算无效位置”

##### 6.3 为什么它特别适合 RNN
因为 RNN 是严格按时间步递推的。

对于一个 batch，如果每条序列长度不一样，  
普通 padding 方案会让 RNN 在每个时间步都处理整批样本，包括很多 `PAD`。

而 `pack_padded_sequence` 会利用“每条序列真实长度”信息，让 RNN 更高效地处理真实部分。

所以这个工具是 PyTorch 中专门为 RNN、LSTM、GRU 这类变长序列模型准备的。


#### 7、mask 和 pack_padded_sequence 的区别是什么

##### 7.1 mask 的思想
mask 的核心是：

先 padding 成规则张量  
再额外提供一个有效位置标记  
后续在需要的时候忽略 `PAD`

也就是说，padding 位置通常还在张量里，  
只是你在某些地方不把它算进去。

##### 7.2 pack_padded_sequence 的思想
pack 的核心是：

利用有效长度，把 padding 后的序列打包  
让 RNN 尽量只处理真实 token

所以 pack 更偏向：

在 RNN 计算阶段就减少 padding 带来的影响和浪费。

##### 7.3 二者的直观区别
mask = “我知道哪些位置无效，后续忽略它们”

pack = “我提前告诉 RNN 真实长度，让它少算甚至不算无效位置”

##### 7.4 它们不是对立关系
这点很重要。

在实际任务中：

- 有时候只用 mask 就够了
- 有时候在 RNN 阶段用 pack，在后续 loss 或 attention 阶段仍然还要配合 mask

所以它们不是互相排斥，而是解决问题的两个层面。


#### 8、一个完整例子来理解有效长度、mask 和 pack

##### 8.1 假设一个 batch 中有三条序列
A：

```python
[2, 3, 4, 0, 0]
```

有效长度 $= 3$

B：

```python
[5, 6, 7, 8, 9]
```

有效长度 $= 5$

C：

```python
[10, 11, 0, 0, 0]
```

有效长度 $= 2$

##### 8.2 对应的 mask
A：

```python
[1, 1, 1, 0, 0]
```

B：

```python
[1, 1, 1, 1, 1]
```

C：

```python
[1, 1, 0, 0, 0]
```

这说明：

- A 只有前 `3` 个位置有效
- B 全部 `5` 个位置有效
- C 只有前 `2` 个位置有效

##### 8.3 如果只做普通 padding + RNN
那么 RNN 仍然会对所有 `5` 个时间步都做计算，  
包括 A 的后两步 `PAD` 和 C 的后三步 `PAD`。

##### 8.4 如果后续使用 mask
那么你可以在：

- 求平均
- 计算 loss
- 做 attention
- 取有效输出

时把这些 `PAD` 位置排除掉。

##### 8.5 如果在 RNN 阶段使用 pack_padded_sequence
那么 RNN 会利用：

```python
[3, 5, 2]
```

这个有效长度信息，  
尽量只对真实 token 做递推处理。

这样既减少无效计算，  
又更有利于得到真实有效的隐藏状态。